# Extended Kalman Filter (EKF) — Demo Notebook

> **Scenario:** A car is driving along a road. A landmark (Statue of Liberty) stands at a known position. The car receives noisy **angle measurements** (bearing φ) to the landmark and uses an EKF to estimate its position and velocity.

---

## Problem Setup

| Symbol | Description | Value |
|--------|-------------|-------|
| `S` | Height of landmark | 20 m |
| `D` | Horizontal position of landmark | 40 m |
| `Δt` | Time step | 0.5 s |
| `u₀` | Control input (acceleration) | -2 m/s² |
| `y₁` | First angle measurement | π/6 rad |

**Initial state distribution:**
$$\hat{\mathbf{x}}_0 \sim \mathcal{N}\left(\begin{bmatrix}0\\5\end{bmatrix}, \begin{bmatrix}0.01 & 0\\0 & 1\end{bmatrix}\right)$$

Where the state is $\mathbf{x} = \begin{bmatrix} p \\ \dot{p} \end{bmatrix}$ (position and velocity).

---
## Motion Model

The car follows a constant-acceleration kinematic model:

$$\mathbf{x}_k = \mathbf{f}(\mathbf{x}_{k-1}, u_{k-1}) + \mathbf{w}_{k-1}$$

$$\mathbf{f}(\mathbf{x}_{k-1}, u_{k-1}) = \begin{bmatrix}1 & \Delta t \\ 0 & 1\end{bmatrix} \mathbf{x}_{k-1} + \begin{bmatrix}\frac{1}{2}\Delta t^2 \\ \Delta t\end{bmatrix} u_{k-1}$$

**Motion model Jacobian** (wrt state):
$$\mathbf{F}_{k-1} = \begin{bmatrix}1 & \Delta t \\ 0 & 1\end{bmatrix}$$

**Motion model Jacobian** (wrt noise):
$$\mathbf{L}_{k-1} = \mathbf{I}_{2 \times 2}$$

---
## Measurement Model

The sensor measures the **bearing angle** φ to the landmark:

$$y_k = h(\mathbf{x}_k) + v_k = \arctan\left(\frac{S}{D - p_k}\right) + v_k$$

**Measurement model Jacobian** (wrt state):
$$\mathbf{H}_k = \begin{bmatrix}\dfrac{S}{(D - \check{p}_k)^2 + S^2} & 0\end{bmatrix} \quad (1 \times 2)$$

**Measurement model Jacobian** (wrt noise):
$$M_k = 1 \quad (\text{scalar})$$

---
## EKF Equations

### Prediction Step
$$\check{\mathbf{x}}_k = \mathbf{F}_{k-1}\,\hat{\mathbf{x}}_{k-1} + \mathbf{B}\, u_{k-1}$$
$$\check{\mathbf{P}}_k = \mathbf{F}_{k-1}\,\hat{\mathbf{P}}_{k-1}\,\mathbf{F}_{k-1}^T + \mathbf{L}_{k-1}\,\mathbf{Q}\,\mathbf{L}_{k-1}^T$$

### Update Step
$$\mathbf{K}_k = \check{\mathbf{P}}_k\,\mathbf{H}_k^T\left(\mathbf{H}_k\,\check{\mathbf{P}}_k\,\mathbf{H}_k^T + M_k\,R\,M_k^T\right)^{-1}$$
$$\hat{\mathbf{x}}_k = \check{\mathbf{x}}_k + \mathbf{K}_k\left(y_k - h(\check{\mathbf{x}}_k)\right)$$
$$\hat{\mathbf{P}}_k = (\mathbf{I} - \mathbf{K}_k\,\mathbf{H}_k)\,\check{\mathbf{P}}_k$$

---
## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---
## Parameters & Initial Conditions

In [ ]:
# Problem parameters
S  = 20.0          # Landmark height [m]
D  = 40.0          # Landmark horizontal position [m]
dt = 0.5           # Time step [s]
u  = -2.0          # Control input: acceleration [m/s^2]

# Initial state mean: [position, velocity]
x_hat = np.array([[0.0],
                  [5.0]])

# Initial state covariance
P_hat = np.array([[0.01, 0.0],
                  [0.0,  1.0]])

# Process noise covariance (assumed)
Q = np.eye(2) * 0.01

# Measurement noise variance (assumed scalar)
R = 0.01

# Measurement at t=1s (k=1, after one step of dt=0.5 ... using at k=2)
y1 = np.pi / 6     # [rad]

---
## Motion Model & Jacobians

In [ ]:
# State transition matrix F
F = np.array([[1, dt],
              [0,  1]])

# Control input matrix B
B = np.array([[0.5 * dt**2],
              [dt]])

# Noise Jacobian L
L = np.eye(2)

def predict(x_hat, P_hat, u):
    """EKF Prediction Step"""
    x_check = F @ x_hat + B * u
    P_check = F @ P_hat @ F.T + L @ Q @ L.T
    return x_check, P_check

print("F =\n", F)
print("B =\n", B)

---
## Measurement Model & Jacobians

In [ ]:
def h(x):
    """Nonlinear measurement function: bearing angle to landmark"""
    p = x[0, 0]
    return np.arctan(S / (D - p))

def H_jacobian(x_check):
    """Jacobian of h wrt state [1x2]"""
    p = x_check[0, 0]
    dh_dp = S / ((D - p)**2 + S**2)
    return np.array([[dh_dp, 0.0]])

def update(x_check, P_check, y):
    """EKF Update Step"""
    H = H_jacobian(x_check)          # 1x2
    M = 1.0                           # scalar

    # Innovation covariance
    S_inn = H @ P_check @ H.T + M * R * M   # scalar

    # Kalman gain: 2x1
    K = P_check @ H.T / S_inn

    # Innovation
    innovation = y - h(x_check)

    # Updated state and covariance
    x_hat_new = x_check + K * innovation
    P_hat_new = (np.eye(2) - K @ H) @ P_check

    return x_hat_new, P_hat_new, K, innovation

---
## Run EKF — Step k=1 (t = 0.5s): Predict Only

In [ ]:
# Step k=1: prediction only (no measurement yet)
x_check_1, P_check_1 = predict(x_hat, P_hat, u)

print("=== Step k=1 (t=0.5s): Prediction ===")
print("Predicted state x_check_1:\n", x_check_1)
print("Predicted covariance P_check_1:\n", P_check_1)

---
## Run EKF — Step k=2 (t = 1.0s): Predict + Update

In [ ]:
# k=1 has no measurement update, so carry forward
x_hat_1 = x_check_1
P_hat_1 = P_check_1

# Step k=2: predict
x_check_2, P_check_2 = predict(x_hat_1, P_hat_1, u)

print("=== Step k=2 (t=1.0s): Prediction ===")
print("Predicted state x_check_2:\n", x_check_2)
print("Predicted covariance P_check_2:\n", P_check_2)

# Step k=2: update with measurement y1
x_hat_2, P_hat_2, K, innov = update(x_check_2, P_check_2, y1)

print("\n=== Step k=2 (t=1.0s): Update ===")
print("Measurement y1 (rad):", y1)
print("Predicted measurement h(x_check_2) (rad):", h(x_check_2))
print("Innovation:", innov)
print("Kalman Gain K:\n", K)
print("Updated state x_hat_2:\n", x_hat_2)
print("Updated covariance P_hat_2:\n", P_hat_2)

---
## Visualization

In [ ]:
times = [0.0, 0.5, 1.0]
positions = [
    x_hat[0, 0],
    x_check_1[0, 0],
    x_hat_2[0, 0]
]
velocities = [
    x_hat[1, 0],
    x_check_1[1, 0],
    x_hat_2[1, 0]
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(times, positions, 'bo-', label='EKF Position Estimate')
axes[0].axvline(1.0, color='r', linestyle='--', label='Measurement at t=1s')
axes[0].set_xlabel('Time [s]')
axes[0].set_ylabel('Position [m]')
axes[0].set_title('Car Position Estimate')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(times, velocities, 'gs-', label='EKF Velocity Estimate')
axes[1].set_xlabel('Time [s]')
axes[1].set_ylabel('Velocity [m/s]')
axes[1].set_title('Car Velocity Estimate')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

---
## Summary

| Time | Step | Position [m] | Velocity [m/s] |
|------|------|-------------|----------------|
| t=0s | Initial | 0.0 | 5.0 |
| t=0.5s | Predict | — | — |
| t=1.0s | Predict + Update | — | — |

> Run the cells above to fill in the table values.

The EKF corrects the purely kinematic prediction using the bearing angle measurement y₁ = π/6, pulling the estimate toward the position consistent with the observed angle to the landmark.